# Normalize the GUD conllu files to Cypriotic Greek

Install dependencies 
- python-docx
- pandas
- openpyxl

Initialize the normalizer

In [1]:
import sys

sys.path.append("CyGr-Normalizer/code/")
from ProcessCyGr import *

In [2]:
# change to your path
PATH = '/Users/maria/Documents/DPMS/cypriotic/cypriot-treebank-project/CyGr-Normalizer/'

rules = ['1-smooth.xlsx', '2-corrections.xlsx', '3-restore.xlsx']
# the first path is just a dummy to initialize the normalizer
far_tool = ProcessCyGr(PATH, PATH + 'spelling-normalizer/', rules)

Normalize the GUD files
- el_gud-ud-train.conllu
- el_gud-ud-test.conllu

In [3]:
from conllu import parse

In [4]:
def normalize_conllu(input_file, output_file):
    with open(input_file, "r", encoding="utf-8") as f:
        conllu_data = f.read()

    sentences = parse(conllu_data)
    for sentence in sentences:
        # Update the sentence-level text metadata if it exists
        if "text" in sentence.metadata:
            norm_text = far_tool.normalize_text(sentence.metadata["text"])
            norm_text = norm_text.replace('\u037e',';').strip()
            norm_text = norm_text.replace("σ’", "σ'")
            norm_text = norm_text.replace("’", "'")
            sentence.metadata["text"] = norm_text
        # Update specific fields for each token
        for token in sentence:
            # skip tokens of σ, σ' 
            if token.get("form") in ["σ", "σ'"]:
                continue
            # Normalize the surface form
            if token.get("form") is not None:
                norm_form = far_tool.normalize_text(token["form"]).rstrip()
                norm_form = norm_form.replace('\u037e',';').strip()
                norm_form = norm_form.replace("’", "'")
                token["form"] = norm_form
            
            # Normalize the lemma (if your normalization affects lemma forms)
            if token.get("lemma") is not None:
                norm_lemma = far_tool.normalize_text(token["lemma"]).rstrip()
                token["lemma"] = norm_lemma.replace('\u037e',';').strip()

    
    with open(output_file, "w", encoding="utf-8") as f:
        f.writelines([sentence.serialize() for sentence in sentences])

In [5]:
normalize_conllu("el_gud-ud-train.conllu", "el_gud-ud-train-normalized.conllu")

In [6]:
normalize_conllu("el_gud-ud-test.conllu", "el_gud-ud-test-normalized.conllu")

In [11]:
import nltk
import conllu
import re

In [66]:
def tokenize_greek_text(text):
    if not text:
        return []
        
    # 1. Διαχωρισμός "στο/στου/στων" -> "σ το/του/των"
    text = re.sub(r'\b([Σσ])τ(ο|ον|ου|ους|ων|η|ην|ης|ες|α|ις)\b', r'\1 τ\2', text)
    
    # 2. ΕΞΑΙΡΕΣΕΙΣ (για λέξεις που ΠΡΕΠΕΙ να σπάσουν με παύλα)
    text = re.sub(r'\b([Κκ]άτω)-([Κκ]άτω)\b', r'\1 - \2', text)
    text = re.sub(r'\b([Κκ]αλού)-([Κκ]ακού)\b', r'\1 - \2', text)
    text = re.sub(r'\b(Αθηνών)-(Πειραιώς)\b', r'\1 - \2', text)
    text = re.sub(r'\b(Αθηνών)-(Κορίνθου)\b', r'\1 - \2', text)
    text = re.sub(r'\b([Γγ]ουλιά)-([Γγ]ουλιά)\b', r'\1 - \2', text)

    # 3. Custom Regex Tokenization
    # ΑΛΛΑΓΗ: Το (?:[-/][\w]+)* έγινε (?:[-/][\w]*)*
    # Αυτό επιτρέπει στην παύλα να υπάρχει ακόμα και αν δεν ακολουθεί λέξη.
    token_pattern = (
        r"[Ττ]ζ[\u0300-\u036F]*['’]|"               # Τζ'
        r"[ΟοΌό],[Ττ][Ιι]|"                         # ό,τι
        r"\.\.\.|"                                  # ...
        r"['’]?[\w\u0300-\u036F]+(?:[-/][\w\u0300-\u036F]*)*['’]?|" 
        r"[^\w\s]"                                  # Στίξη
    )
    
    return re.findall(token_pattern, text)

In [67]:
def fix_conllu(input_file, output_file):
    with open(input_file, 'r', encoding='utf-8') as f_in, open(output_file, 'w', encoding='utf-8') as f_out:
        for sentence in conllu.parse_incr(f_in):
            text = sentence.metadata.get("text")
            #text_tokens = nltk.word_tokenize(text)
            text_tokens = tokenize_greek_text(text)
            token_idx = 0
            for token in sentence:
                if type(token["id"]) is not int:
                    continue

                if token_idx >= len(text_tokens):
                    break

                sentence_word = text_tokens[token_idx]

                if token['form'] != sentence_word:
                    token["form"] = sentence_word

               
                token_idx += 1
               
            f_out.write(sentence.serialize())
        

In [68]:
fix_conllu("el_gud-ud-train-normalized.conllu", "el_gud-ud-train-normalized-postprocessed.conllu")

In [69]:
fix_conllu("el_gud-ud-test-normalized.conllu", "el_gud-ud-test-normalized-postprocessed.conllu")